In [1]:
import json
import re
import os
from pathlib import Path
from PIL import Image
import pdfplumber
from transformers import pipeline
import torch
import cv2
import numpy as np
# from pipe_fn import pipe
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,
    calculate_model_confidence,
)



# =========================
# MODEL INITIALIZATION
# =========================
pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)



# =========================
# PDF TABLE CROPPING
# =========================
def crop_claim_tables(pdf_path, output_dir="cropped_tables"):
    os.makedirs(output_dir, exist_ok=True)

    cropped_images = []  # store output paths

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            words = page.extract_words()

            start_y = None
            end_y = None

            # Find table start and end
            for w in words:
                text = w["text"].strip()

                # Start from Claim #
                if "Claim" in text and start_y is None:
                    start_y = w["top"] - 10

                # End at Total row
                if text == "Total":
                    end_y = w["bottom"] + 10

            if start_y and end_y and end_y > start_y:

                bbox = (
                    0,
                    start_y,
                    page.width,
                    end_y
                )

                cropped = page.crop(bbox)

                output_file = os.path.join(
                    output_dir,
                    f"page_{page_num}_table.png"
                )

                cropped.to_image(resolution=300).save(
                    output_file,
                    format="PNG"
                )

                expected_rows = count_service_rows(
                    page,
                    start_y,
                    end_y
                )

                cropped_images.append({
                    "image_path": output_file,
                    "expected_rows": expected_rows
                })

                print(
                    f"Page {page_num}: saved -> {output_file}"
                )

            else:
                print(
                    f"Page {page_num}: table not found"
                )

    return cropped_images


def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()

    service_rows = set()

    date_pattern = re.compile(
        r"\d{2}/\d{2}-\d{2}/\d{2}/\d{4}"
    )

    for w in words:

        text = w["text"].strip()

        if date_pattern.fullmatch(text):

            y = float(w["top"])

            if region_top <= y <= region_bottom:

                service_rows.add(round(y, 1))

    return len(service_rows)


# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img  = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
    sums  = np.sum(thresh, axis=1)
    th    = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]
    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)
    return img


# =========================
# AMOUNT CONVERSION
# =========================
def convert_amounts_to_string(obj):
    amount_fields = {
        "total_charge", "not_covered", "provider_discount",
        "deductible_copay", "balance", "Benefit",
    }
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            if k in amount_fields:
                try:
                    new_obj[k] = f"{float(str(v).replace('$','').replace(',','').strip()):.2f}"
                except Exception:
                    new_obj[k] = ""
            else:
                new_obj[k] = convert_amounts_to_string(v)
        return new_obj
    elif isinstance(obj, list):
        return [convert_amounts_to_string(i) for i in obj]
    else:
        return obj


# =========================
# PROMPT BUILDER
# =========================
def build_prompt(pdf_name: str):
    return f"""
Extract dental EOB claim table data from the image.

RULES

1. Extract:
   - patient_name
   - relationship

2. Extract ALL visible service rows.

3. Preserve row order.

4. Do NOT skip duplicate rows.

5. Ignore:
   - Tooth Number
   - See Comments
   - Pay %

6. Extract all remaining columns.

7. Extract the Total row into a separate totals object.

8. Return ONLY valid JSON.
OUTPUT FORMAT

{{
  "patient_name": {{
    "value": "",
    "confidence": 0.0
  }},

  "relationship": {{
    "value": "",
    "confidence": 0.0
  }},

  "services": [
    {{
      "service_date": {{
        "value": "",
        "confidence": 0.0
      }},
      "total_charge": {{
        "value": "",
        "confidence": 0.0
      }},
      "not_covered": {{
        "value": "",
        "confidence": 0.0
      }},
      "provider_discount": {{
        "value": "",
        "confidence": 0.0
      }},
      "deductible_copay": {{
        "value": "",
        "confidence": 0.0
      }},
      "balance": {{
        "value": "",
        "confidence": 0.0
      }},
      "Benefit": {{
        "value": "",
        "confidence": 0.0
      }}
    }}
  ],

  "totals": {{
    "total_charge": {{
      "value": "",
      "confidence": 0.0
    }},
    "not_covered": {{
      "value": "",
      "confidence": 0.0
    }},
    "provider_discount": {{
      "value": "",
      "confidence": 0.0
    }},
    "deductible_copay": {{
      "value": "",
      "confidence": 0.0
    }},
    "balance": {{
      "value": "",
      "confidence": 0.0
    }},
    "Benefit": {{
      "value": "",
      "confidence": 0.0
    }}
  }}
}}


 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.
"""


# =========================
# AMOUNT HELPERS
# =========================
def parse_amount(x) -> float:
    if x is None or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace("$", "").replace(",", "").strip())
    except ValueError:
        return 0.0
    

def normalize_service_dates(obj):
    """
    Convert:
        02/18-02/18/2026  -> 02/18/2026
        08/06-08/06/2025  -> 08/06/2025

    Leaves other formats unchanged.
    """
    pattern = re.compile(r"^\d{2}/\d{2}-(\d{2}/\d{2}/\d{4})$")

    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ("service_date", "service_dates") and isinstance(v, str):
                m = pattern.match(v.strip())
                if m:
                    obj[k] = m.group(1)
            else:
                normalize_service_dates(v)

    elif isinstance(obj, list):
        for item in obj:
            normalize_service_dates(item)

    return obj

def compute_totals_from_services(services: list) -> dict:
    fields = [
        "total_charge",
        "not_covered",
        "provider_discount",
        "deductible_copay",
        "balance",
        "Benefit",
    ]
    return {
        f: round(sum(parse_amount(s.get(f, "")) for s in services), 2)
        for f in fields
    }


def services_are_empty(services: list) -> bool:
    """Return True when every amount field in every service row is blank/zero."""
    amount_fields = [
        "total_charge",
        "not_covered",
        "provider_discount",
        "deductible_copay",
        "balance",
        "Benefit",
    ]
    for svc in services:
        for f in amount_fields:
            if str(svc.get(f, "")).strip() not in ("", "0.00", "0"):
                return False
    return True


# =========================
# VALIDATION
# =========================
def validate_patient_totals(
    patient: dict,
    patient_name: str,
    expected_row_count: int
):
    services = patient.get("services", [])
    totals = patient.get("totals", {})

    if not services:
        return False, "No services found", [{"error": "empty_services"}]

    if services_are_empty(services):
        msg = "All service rows are empty — model likely failed to extract data"
        print(f"\n❌ {msg}")
        return False, msg, [{"error": "all_service_rows_empty"}]

    computed_totals = compute_totals_from_services(services)

    errors = []
    has_error = False

    DISPLAY_NAMES = {
        "total_charge": "Total Charge",
        "not_covered": "Not Covered",
        "provider_discount": "Provider Discount",
        "deductible_copay": "Deductible/Copay",
        "balance": "Balance",
        "Benefit": "Benefit",
    }

    # =========================================================
    # ROW COUNT VALIDATION
    # =========================================================
    extracted_row_count = len(services)

    row_match = expected_row_count == extracted_row_count

    print(f"\n📊 Row Count Validation [{patient_name}]")
    print("-" * 75)

    print(
        f"{'✅' if row_match else '❌'} "
        f"row_count detected={expected_row_count:<5} "
        f"| extracted={extracted_row_count:<5} "
        f"{'match' if row_match else 'mismatch'}"
    )

    print("-" * 75)

    if not row_match:
        has_error = True
        errors.append({
            "type": "row_count_mismatch",
            "expected_rows": expected_row_count,
            "extracted_rows": extracted_row_count
        })

    # =========================================================
    # TOTALS VALIDATION
    # =========================================================
    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 75)

    for field, computed_value in computed_totals.items():

        extracted_value = round(
            parse_amount(totals.get(field, "")),
            2
        )

        diff = round(computed_value - extracted_value, 2)

        match = abs(diff) <= 0.01

        if not match:
            has_error = True

            errors.append({
                "type": "field_mismatch",
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value,
                "difference": diff,
            })

        display_name = DISPLAY_NAMES.get(field, field)

        print(
            f"{'✅' if match else '❌'} "
            f"{display_name:25s} "
            f"computed={computed_value:<10.2f} "
            f"| extracted={extracted_value:<10.2f} "
            f"{'match' if match else 'mismatch'}"
        )

    print("-" * 75)

    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, "", errors

    print(f"✅ [{patient_name}] Validation PASSED\n")
    return True, "", []


# =========================
# JSON CLEANER
# =========================
def extract_json(text: str) -> dict:
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in model output")
    return json.loads(text[start:end])


def save_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# =========================
# DENIAL CHECK
# =========================
def check_claim_denied(pdf_path: str) -> str:
    denial_keywords = ["denied", "denial"]
    stop_phrase     = "Comments"

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text = page.extract_text()
            if not full_text:
                continue
            searchable = full_text.lower()
            if stop_phrase.lower() in searchable:
                searchable = searchable.split(stop_phrase.lower())[0]
            for keyword in denial_keywords:
                if keyword in searchable:
                    print(f"claim denied keyword found: {keyword}")
                    return "denied"

    return "not denied"


# =========================
# RETRY HELPER
# =========================
MAX_RETRIES = 3

def run_model_with_retry(image: Image.Image, prompt: str, expected_rows: int):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  prompt},
            ],
        }
    ]

    last_parsed = None
    last_model_confidence = None

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  🔄 Model attempt {attempt}/{MAX_RETRIES}")

        with torch.no_grad():
            output = pipe(
                messages,
                max_new_tokens=15000,
                temperature=0.0,
                do_sample=False,
                top_k=1,
                top_p=1.0
            )

        raw = output[0]["generated_text"]
        if isinstance(raw, list):
            raw = raw[-1]["content"]

        try:
            raw_parsed = extract_json(raw)
        except Exception as e:
            print(f"  ❌ JSON parse failed on attempt {attempt}: {e}")
            continue

        # Compute per-field confidence off the raw {value, confidence}
        # structure BEFORE unwrapping, then flatten to plain values so
        # every downstream step (amount conversion, date normalization,
        # totals validation) keeps working on plain strings like before.
        model_confidence = calculate_model_confidence(raw_parsed)
        parsed = _unwrap_vlm_output(raw_parsed)

        services = parsed.get("services", [])
        row_ok   = (len(services) == expected_rows)
        empty_ok = not services_are_empty(services)

        if row_ok and empty_ok:
            print(f"  ✅ Accepted on attempt {attempt}")
            parsed["_model_confidence"] = model_confidence
            return parsed

        print(
            f"  ⚠️  attempt {attempt}: rows={len(services)} (expected {expected_rows}), "
            f"all_empty={not empty_ok}"
        )
        last_parsed = parsed
        last_model_confidence = model_confidence

    print(f"  ⚠️  All {MAX_RETRIES} attempts exhausted — using last result")
    if last_parsed is not None:
        last_parsed["_model_confidence"] = last_model_confidence
    return last_parsed


# =========================
# MAIN PIPELINE
# =========================
def run_pipeline(pdf_path: str, output_dir: str = "EOB_OUTPUT/Bestlife", company_name = "Bestlife"):
    # pdf_name is used as eob_id throughout
    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)

    base_dir         = os.path.join(output_dir, pdf_name)
    cropped_dir      = os.path.join(base_dir, "cropped_images")
    json_output_path = os.path.join(base_dir, f"{pdf_name}_output.json")

    if os.path.exists(json_output_path):
        print(f"⏭️  Skipping {pdf_name} — output already exists")
        return None

    os.makedirs(base_dir,    exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    image_paths  = crop_claim_tables(pdf_path, output_dir=cropped_dir)
    print(type(image_paths))
    print(image_paths[:3])
    is_denied    = check_claim_denied(pdf_path)
    print(f"claim status: {is_denied}")

    # Build prompt once with pdf_name as eob_id
    final_prompt = build_prompt(pdf_name)

    results = []

    for idx, item in enumerate(image_paths):
        img_path      = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"\nProcessing table {idx+1}/{len(image_paths)}")

        image_cv  = make_table(img_path)
        image_pil = Image.fromarray(image_cv).convert("RGB")

        parsed = run_model_with_retry(image_pil, final_prompt, expected_rows)

        if parsed is None:
            print(f"❌ Skipping table {idx+1} — model returned nothing usable")
            continue

        # Normalise relationship key spelling
        if "Realationship" in parsed:
            parsed["relationship"] = parsed.pop("Realationship")

        parsed = convert_amounts_to_string(parsed)
        parsed = normalize_service_dates(parsed)

        parsed["_expected_rows"] = expected_rows

        print("Extracted:")
        print(json.dumps(parsed, indent=2))

        results.append(parsed)

    # Validate every patient
    for patient in results:
        is_valid, log, errors = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("patient_name", "UNKNOWN"),
            expected_row_count=patient.get("_expected_rows", 0),
        )
        patient["validation"] = {
            "status": is_valid,
            "errors": errors,
        }

    # Overall document confidence, computed from each patient's
    # per-field model confidence before we strip the internal keys.
    confidence_results = list(results)
    confidence_score = calculate_eob_confidence(confidence_results)

    for patient in results:
        patient.pop("_expected_rows", None)
        patient.pop("_model_confidence", None)

    final = [
        {
            "eob_id":       pdf_name,
            "file_name":pdf_full_name,
            "claim_status": is_denied,
            "confidence_score": confidence_score,
            "Generated_CDT_code": True,
            "payor": "Bestlife",
            "patients":     results,
        }
    ]

    success_path, failed_path = save_split_output(
                final,
                company_name=company_name,
                pdf_name=pdf_name,
                pdf_path=pdf_path,
                cropped_dir=cropped_dir,
            )
        
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 18:03:05.719000 3375491 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 18:03:05.734000 3375491 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

## Test 1

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Bestlife/pdfs/Pmt_EOP_889240221.pdf")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `top_k` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `top_p` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Page 1: saved -> EOB_OUTPUT/Bestlife/889240221/cropped_images/page_1_table.png
<class 'list'>
[{'image_path': 'EOB_OUTPUT/Bestlife/889240221/cropped_images/page_1_table.png', 'expected_rows': 4}]
claim status: not denied

Processing table 1/1
  🔄 Model attempt 1/3


[transformers] Both `max_new_tokens` (=15000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Accepted on attempt 1
Extracted:
{
  "patient_name": "Thi Tran",
  "relationship": "MEMBER",
  "services": [
    {
      "service_date": "05/04/2026",
      "total_charge": "148.00",
      "not_covered": "0.00",
      "provider_discount": "82.00",
      "deductible_copay": "0.00",
      "balance": "66.00",
      "Benefit": "66.00"
    },
    {
      "service_date": "05/04/2026",
      "total_charge": "104.00",
      "not_covered": "0.00",
      "provider_discount": "58.00",
      "deductible_copay": "0.00",
      "balance": "46.00",
      "Benefit": "46.00"
    },
    {
      "service_date": "05/04/2026",
      "total_charge": "49.00",
      "not_covered": "0.00",
      "provider_discount": "30.00",
      "deductible_copay": "0.00",
      "balance": "19.00",
      "Benefit": "19.00"
    },
    {
      "service_date": "05/04/2026",
      "total_charge": "40.00",
      "not_covered": "0.00",
      "provider_discount": "25.00",
      "deductible_copay": "0.00",
      "balance": "15.00

[{'eob_id': '889240221',
  'file_name': 'Pmt_EOP_889240221.pdf',
  'claim_status': 'not denied',
  'confidence_score': 100.0,
  'Generated_CDT_code': True,
  'payor': 'Bestlife',
  'patients': [{'patient_name': 'Thi Tran',
    'relationship': 'MEMBER',
    'services': [{'service_date': '05/04/2026',
      'total_charge': '148.00',
      'not_covered': '0.00',
      'provider_discount': '82.00',
      'deductible_copay': '0.00',
      'balance': '66.00',
      'Benefit': '66.00'},
     {'service_date': '05/04/2026',
      'total_charge': '104.00',
      'not_covered': '0.00',
      'provider_discount': '58.00',
      'deductible_copay': '0.00',
      'balance': '46.00',
      'Benefit': '46.00'},
     {'service_date': '05/04/2026',
      'total_charge': '49.00',
      'not_covered': '0.00',
      'provider_discount': '30.00',
      'deductible_copay': '0.00',
      'balance': '19.00',
      'Benefit': '19.00'},
     {'service_date': '05/04/2026',
      'total_charge': '40.00',
      'no

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Bestlife/pdfs/Pmt_EOP_850189507.pdf")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `top_k` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `top_p` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Page 1: saved -> EOB_OUTPUT/Bestlife/850189507/cropped_images/page_1_table.png
<class 'list'>
[{'image_path': 'EOB_OUTPUT/Bestlife/850189507/cropped_images/page_1_table.png', 'expected_rows': 5}]
claim status: not denied

Processing table 1/1
  🔄 Model attempt 1/3


[transformers] Both `max_new_tokens` (=15000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Accepted on attempt 1
Extracted:
{
  "patient_name": "Laura Perry",
  "relationship": "SPOUSE",
  "services": [
    {
      "service_date": "03/03/2026",
      "total_charge": "159.00",
      "not_covered": "0.00",
      "provider_discount": "74.00",
      "deductible_copay": "0.00",
      "balance": "85.00",
      "Benefit": "85.00"
    },
    {
      "service_date": "03/03/2026",
      "total_charge": "104.00",
      "not_covered": "0.00",
      "provider_discount": "58.00",
      "deductible_copay": "0.00",
      "balance": "46.00",
      "Benefit": "46.00"
    },
    {
      "service_date": "03/03/2026",
      "total_charge": "95.00",
      "not_covered": "0.00",
      "provider_discount": "57.00",
      "deductible_copay": "0.00",
      "balance": "38.00",
      "Benefit": "38.00"
    },
    {
      "service_date": "03/03/2026",
      "total_charge": "49.00",
      "not_covered": "0.00",
      "provider_discount": "30.00",
      "deductible_copay": "0.00",
      "balance": "19

[{'eob_id': '850189507',
  'claim_status': 'not denied',
  'confidence_score': 100.0,
  'Generated_CDT_code': True,
  'payor': 'Bestlife',
  'patients': [{'patient_name': 'Laura Perry',
    'relationship': 'SPOUSE',
    'services': [{'service_date': '03/03/2026',
      'total_charge': '159.00',
      'not_covered': '0.00',
      'provider_discount': '74.00',
      'deductible_copay': '0.00',
      'balance': '85.00',
      'Benefit': '85.00'},
     {'service_date': '03/03/2026',
      'total_charge': '104.00',
      'not_covered': '0.00',
      'provider_discount': '58.00',
      'deductible_copay': '0.00',
      'balance': '46.00',
      'Benefit': '46.00'},
     {'service_date': '03/03/2026',
      'total_charge': '95.00',
      'not_covered': '0.00',
      'provider_discount': '57.00',
      'deductible_copay': '0.00',
      'balance': '38.00',
      'Benefit': '38.00'},
     {'service_date': '03/03/2026',
      'total_charge': '49.00',
      'not_covered': '0.00',
      'provider_d